# INF-HARTO · Análisis del estudio con jugadores

Cuaderno del análisis descrito en la sección «Análisis de datos» del capítulo 7 de la memoria.

**Cómo usarlo**
1. En la hoja de respuestas del formulario de telemetría: *Archivo → Descargar → Valores separados por comas (.csv)*.
2. (Opcional) Lo mismo con las respuestas del cuestionario final de Google Forms.
3. *Entorno de ejecución → Ejecutar todo*. Si los archivos no están subidos, el cuaderno los pide.
4. Al final se descarga `datos_por_recorrido.csv`, con un valor por participante y recorrido.

El análisis es descriptivo (medianas y conteos), ya que con tan pocos participantes no se aplican pruebas estadísticas.

La clave de verificación (HMAC) **no** está en el cuaderno: se pide al ejecutar. Si se deja vacía, no se comprueban las firmas.

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────
ARCHIVO_TELEMETRIA = 'telemetria.csv'      # CSV (o TSV) descargado de la hoja de respuestas
ARCHIVO_FINAL      = 'cuestionario_final.csv'  # opcional: respuestas del cuestionario final
VERSION_ESTUDIO    = 'estudio-5'           # solo se usan las filas de esta versión (None = todas)
CONFIG_A, CONFIG_B = 'Permisiva', 'Exigente'

ITEMS = {1: 'Disfrute', 2: 'Control', 3: 'Concentración', 4: 'Sincronía con la música',
         5: 'Desafío percibido', 6: 'Atención al ritmo', 7: 'Justicia de los errores', 8: 'Seguir jugando'}

In [ ]:
import os, io, hmac, hashlib, getpass
import numpy as np, pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 60)

EN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
def pedir(nombre, obligatorio=True):
    if os.path.exists(nombre) or not EN_COLAB: return nombre if os.path.exists(nombre) else None
    print(f'Sube {nombre}' + ('' if obligatorio else ' (opcional: pulsa Cancelar para saltarlo)'))
    from google.colab import files
    try: subidos = files.upload()
    except Exception: subidos = {}
    return next(iter(subidos), None)

ARCHIVO_TELEMETRIA = pedir(ARCHIVO_TELEMETRIA)
CLAVE = (os.environ.get('CLAVE_ESTUDIO') or (getpass.getpass('Clave de verificación (vacío = no comprobar): ') if EN_COLAB else '')).strip()

## 1. Lectura de las filas

In [ ]:
CAB_INTENTO = ('tipo;participante;fecha_utc;version;parte;nivel;config;orden;intento;resultado;duracion_s;beats_alcanzados;'
  'precision_sync;latidos;aciertos;perfect;good;fallos;mejor_racha;desfase_medio_ms;desfase_abs_medio_ms;desfase_sd_ms;'
  'energia_gastada_acciones;energia_perdida_fallos;energia_perdida_azul;energia_recuperada;energia_media;energia_min;'
  'tiempo_azul_s;entradas_azul;saltos;dashes;slides;wallruns;monedas;mejoras_equipadas;'
  'monedas_recogidas;monedas_nivel_total;contratos_activos;contratos_fallados;ronda;obstaculo;orden_obstaculos').split(';')
CABECERAS = {
  'intento': CAB_INTENTO,
  'cuestionario': 'tipo;participante;fecha_utc;version;parte;nivel;config;orden;p1;p2;p3;p4;p5;p6;p7;p8;preferencia'.split(';'),
  'contrato': 'tipo;participante;fecha_utc;version;parte;nivel;evento;contrato;dificultad;recompensa;monedas'.split(';'),
  'tienda': 'tipo;participante;fecha_utc;version;parte;nivel;evento;objeto;precio;monedas_restantes;muescas_usadas;muescas_total;duracion_s'.split(';'),
  'abandono': 'tipo;participante;fecha_utc;version;parte;nivel;intentos'.split(';'),
  'salto_seccion': 'tipo;participante;fecha_utc;version;parte;nivel;intentos;seccion;obstaculo'.split(';'),
}
COLA = ['fila', 'verificacion']

sep = '\t' if open(ARCHIVO_TELEMETRIA, encoding='utf-8').readline().count('\t') else ','
crudo = pd.read_csv(ARCHIVO_TELEMETRIA, sep=sep, dtype=str, keep_default_na=False)
lineas = crudo.iloc[:, -1].str.strip()          # última columna = la fila de datos
lineas = lineas[lineas.str.contains(';')]

def firma_ok(linea):
    cuerpo, _, firma = linea.rpartition(';')
    mac = hmac.new(CLAVE.encode('utf-8'), cuerpo.encode('utf-8'), hashlib.sha256).hexdigest()[:8]
    return mac == firma

filas = {t: [] for t in CABECERAS}; descartes = {'tipo desconocido': 0, 'nº de campos': 0, 'firma': 0}
for l in lineas:
    c = l.split(';'); t = c[0]
    if t not in CABECERAS: descartes['tipo desconocido'] += 1; continue
    cab = CABECERAS[t] + COLA
    if t == 'salto_seccion' and len(c) != len(cab):          # (campos variables según la versión)
        cab = (CABECERAS[t][:len(c) - 2] + COLA) if len(c) - 2 <= len(CABECERAS[t]) else None
    if cab is None or len(c) != len(cab): descartes['nº de campos'] += 1; continue
    if CLAVE and not firma_ok(l): descartes['firma'] += 1; continue
    filas[t].append(dict(zip(cab, c)))

D = {t: pd.DataFrame(v, columns=(CABECERAS[t] + COLA) if not v else None) for t, v in filas.items()}
for t, df in D.items():
    if df.empty: continue
    if VERSION_ESTUDIO: D[t] = df = df[df.version == VERSION_ESTUDIO]
    D[t] = df.drop_duplicates(['participante', 'fila'])
    D[t]['fila'] = pd.to_numeric(D[t]['fila'])
print('Filas leídas:', {t: len(df) for t, df in D.items()}); print('Descartadas:', descartes)

In [ ]:
num_int = ['latidos','aciertos','perfect','good','fallos','mejor_racha','entradas_azul','saltos','dashes','slides','wallruns',
           'monedas','monedas_recogidas','monedas_nivel_total','intento']
num_flt = ['duracion_s','beats_alcanzados','precision_sync','desfase_medio_ms','desfase_abs_medio_ms','desfase_sd_ms',
           'energia_gastada_acciones','energia_perdida_fallos','energia_perdida_azul','energia_recuperada','energia_media',
           'energia_min','tiempo_azul_s']
I = D['intento'].copy()
for col in num_int + num_flt: I[col] = pd.to_numeric(I[col], errors='coerce')
Q = D['cuestionario'].copy()
for k in range(1, 9): Q[f'p{k}'] = pd.to_numeric(Q[f'p{k}'], errors='coerce')

## 2. Participantes válidos

In [ ]:
codigos = sorted(set().union(*[set(df.participante) for df in D.values() if not df.empty]))
pruebas   = {c for c in codigos if c.upper().startswith('TEST-')}
abandonos = set(D['abandono'].participante) if not D['abandono'].empty else set()
q_partes  = Q.groupby('participante').parte.apply(set)
completos = {c for c, p in q_partes.items() if {'1', '2'} <= p}
validos   = sorted(completos - pruebas - abandonos)

resumen = pd.DataFrame({'código': codigos})
resumen['estado'] = resumen['código'].map(lambda c: 'prueba (TEST-)' if c in pruebas else 'abandonó' if c in abandonos
                                       else 'válido' if c in validos else 'incompleto')
display(resumen.value_counts('estado').rename('participantes').to_frame())
display(resumen)
orden = Q[Q.participante.isin(validos)].groupby('participante').orden.first()
print(f'Válidos: {len(validos)}  ·  orden AB: {(orden == "AB").sum()}  ·  orden BA: {(orden == "BA").sum()}')

## 3. Un valor por participante y recorrido

In [ ]:
E = I[I.participante.isin(validos) & I.parte.isin(['1', '2'])]
saltos = (D['salto_seccion'].groupby(['participante', 'parte']).size().rename('secciones_saltadas')
          if not D['salto_seccion'].empty else pd.Series(dtype=int, name='secciones_saltadas'))

def por_recorrido(g):
    lat = g.latidos.sum()
    return pd.Series({
        'config': g.config.iloc[0], 'orden': g.orden.iloc[0],
        'intentos': len(g), 'muertes': g.resultado.str.startswith('muerte').sum(),
        'completado': (g.resultado == 'completado').any(),
        'precision': g.aciertos.sum() / lat if lat else np.nan,                          # aciertos / clics
        'perfect_pct': g.perfect.sum() / lat if lat else np.nan,
        'desfase_abs_ms': np.average(g.desfase_abs_medio_ms, weights=g.latidos) if lat else np.nan,
        'energia_min': g.energia_min.min(),
        'tiempo_s': g.duracion_s.sum(),
    })
R = E.groupby(['participante', 'parte']).apply(por_recorrido, include_groups=False)
R = R.join(saltos).fillna({'secciones_saltadas': 0})
R['saltó_sección'] = R.secciones_saltadas > 0
qq = Q[Q.participante.isin(validos)].set_index(['participante', 'parte'])[[f'p{k}' for k in range(1, 9)] + ['preferencia']]
R = R.join(qq).reset_index()
R['grupo'] = R.config.map({CONFIG_A: 'A', CONFIG_B: 'B'})
display(R.round(3))

## 4. Comparación A (permisiva) frente a B (exigente) · PI1, PI2 · H1–H4

In [ ]:
ancho = R.pivot(index='participante', columns='grupo')
variables = [(f'p{k}', f'Ítem {k}: {v}') for k, v in ITEMS.items()] + [
    ('precision', 'Precisión rítmica'), ('muertes', 'Muertes'), ('energia_min', 'Energía mínima'),
    ('desfase_abs_ms', 'Desfase absoluto (ms)')]

def comparar(ancho):
    # Para cada variable: mediana con A y con B, y cuántos participantes suben, bajan o se mantienen de A a B.
    out = []
    for v, nombre in variables:
        a, b = ancho[(v, 'A')].astype(float), ancho[(v, 'B')].astype(float)
        d = (b - a).round(6).dropna()
        out.append({'variable': nombre, 'n': len(d), 'mediana A': a.median(), 'mediana B': b.median(),
                    'suben': int((d > 0).sum()), 'bajan': int((d < 0).sum()), 'iguales': int((d == 0).sum())})
    return pd.DataFrame(out)

T = comparar(ancho)
display(T.round(3))


In [ ]:
# Sensibilidad: repetir sin los participantes que se saltaron alguna sección
con_salto = set(R.loc[R['saltó_sección'], 'participante'])
if con_salto:
    print('Sin', sorted(con_salto))
    display(comparar(ancho.drop(index=list(con_salto), errors='ignore')).round(3))
else:
    print('Nadie se saltó secciones: no hace falta repetir el análisis.')

## 5. Efecto del orden

In [ ]:
uno, dos = R.pivot(index='participante', columns='parte', values='precision').reindex(columns=['1', '2']).T.values
mejoran = int(np.nansum(dos > uno))
print(f'Precisión, recorrido 1 → 2: mediana {np.nanmedian(uno):.3f} → {np.nanmedian(dos):.3f}; mejoran {mejoran} de {len(uno)}')
for k in (1, 5):
    a, b = R.pivot(index='participante', columns='parte', values=f'p{k}').reindex(columns=['1', '2']).T.values
    print(f'Ítem {k} ({ITEMS[k]}), recorrido 1 → 2: mediana {np.nanmedian(a):.1f} → {np.nanmedian(b):.1f}')


## 6. Preferencia · PI3

In [ ]:
def pref_config(row):
    p = str(row.preferencia).strip().lower()
    if p in (CONFIG_A.lower(), CONFIG_B.lower()): return 'A' if p == CONFIG_A.lower() else 'B'
    if p == 'igual': return 'igual'
    if p in ('anterior', 'esta'):                      # (versiones antiguas: relativo al recorrido)
        uno = R[(R.participante == row.participante) & (R.parte == '1')].grupo.iloc[0]
        return uno if p == 'anterior' else ({'A': 'B', 'B': 'A'}[uno])
    return np.nan
P = R[R.parte == '2'][['participante', 'preferencia']].copy()
P['prefiere'] = P.apply(pref_config, axis=1)
cuenta = P.prefiere.value_counts().reindex(['A', 'B', 'igual'], fill_value=0); display(cuenta.to_frame('participantes'))

mejor = ancho['precision'].idxmax(axis=1).rename('mejor_precisión')
P = P.set_index('participante').join(mejor)
P['coincide'] = np.where(P.prefiere.isin(['A', 'B']), P.prefiere == P['mejor_precisión'], np.nan)
display(P)
print(f'Prefieren la configuración en la que fueron más precisos: {int(np.nansum(P.coincide.astype(float)))} de {int(P.coincide.notna().sum())}')

## 7. Precisión y *flow* · PI4 · H5

In [ ]:
M = R.groupby('participante')[['precision', 'p3', 'p4']].mean()      # un valor por participante
display(M.sort_values('precision', ascending=False).round(3))
for k in (3, 4):
    print(f'Ítem {k} ({ITEMS[k]}): respuestas', R[f'p{k}'].value_counts().sort_index().to_dict())


## 8. Modo normal · PI5 (descriptivo)

In [ ]:
L  = I[(I.parte == 'libre') & I.participante.isin(validos)]
Ti = D['tienda'];   Ti = Ti[(Ti.parte == 'libre') & Ti.participante.isin(validos)] if not Ti.empty else Ti
Co = D['contrato']; Co = Co[(Co.parte == 'libre') & Co.participante.isin(validos)] if not Co.empty else Co
print(f'Siguieron jugando: {L.participante.nunique()} de {len(validos)}')
if not L.empty:
    display(L.groupby('participante').agg(intentos=('intento', 'size'), minutos=('duracion_s', lambda s: s.sum() / 60),
                                          nivel_max=('nivel', 'max'), monedas_recogidas=('monedas_recogidas', 'sum')).round(1))
if not Ti.empty:
    print('Compras:'); display(Ti[Ti.evento == 'compra'].groupby('objeto').size().sort_values(ascending=False).to_frame('compras'))
if not Co.empty:
    print('Contratos:'); display(Co.groupby(['contrato', 'evento']).size().unstack(fill_value=0))

## 9. Cuestionario final (opcional)

In [ ]:
ARCHIVO_FINAL = pedir(ARCHIVO_FINAL, obligatorio=False)
if ARCHIVO_FINAL:
    F = pd.read_csv(ARCHIVO_FINAL, dtype=str)
    col_cod = next((c for c in F.columns if 'código' in c.lower() or 'codigo' in c.lower()), None)
    if col_cod:
        F[col_cod] = F[col_cod].str.strip().str.upper()
        print(f'Respuestas: {len(F)} · de participantes válidos: {F[col_cod].isin(validos).sum()}')
        F = F[F[col_cod].isin(validos)]
    for c in F.columns[1:]:
        vc = F[c].value_counts(dropna=True)
        if 1 < len(vc) <= 8: display(vc.to_frame(c[:70]))
else:
    print('Sin cuestionario final.')

## 10. Datos por recorrido


In [ ]:
R.to_csv('datos_por_recorrido.csv', index=False)
if EN_COLAB:
    from google.colab import files; files.download('datos_por_recorrido.csv')
